# 🔄 Flussi di Lavoro Base per Agenti con Microsoft Foundry (Python)

## 📋 Tutorial di Orchestrazione del Workflow

Questo notebook introduce le potenti funzionalità del **Workflow Builder** del Microsoft Agent Framework. Impara a creare flussi di lavoro sofisticati e multi-step per agenti che possono gestire processi aziendali complessi e coordinare senza problemi più operazioni AI.

> **Nota sulla migrazione:** Questo esempio faceva precedentemente riferimento ai Modelli GitHub. I Modelli GitHub sono deprecati (in pensionamento a luglio 2026), quindi ora viene utilizzato **Microsoft Foundry** tramite il `FoundryChatClient`, che usa l'API **Responses** di Azure OpenAI.

## 🎯 Obiettivi di Apprendimento

### 🏗️ **Architettura del Workflow**
- **Workflow Builder**: Progetta e orchestra processi complessi multi-step
- **Esecuzione Basata su Eventi**: Gestisci eventi e transizioni di stato del workflow
- **Design Visuale del Workflow**: Crea e visualizza strutture di workflow
- **Integrazione Microsoft Foundry**: Sfrutta modelli AI all’interno di contesti workflow

### 🔄 **Orchestrazione dei Processi**
- **Operazioni Sequenziali**: Collega più task agent in ordine logico
- **Logica Condizionale**: Implementa punti decisionali e flussi ramificati
- **Gestione degli Errori**: Recupero robusto dagli errori e resilienza del workflow
- **Gestione dello Stato**: Monitora e gestisci lo stato di esecuzione del workflow

### 📊 **Modelli di Workflow Aziendali**
- **Automazione dei Processi Aziendali**: Automatizza flussi di lavoro organizzativi complessi
- **Coordinamento Multi-Agente**: Coordina più agenti specializzati
- **Esecuzione Scalabile**: Progetta workflow per operazioni su scala enterprise
- **Monitoraggio & Osservabilità**: Monitora prestazioni e risultati dei workflow

## ⚙️ Prerequisiti & Configurazione

### 📦 **Dipendenze Richieste**

Installa l'Agent Framework con capacità di workflow:

```bash
pip install agent-framework -U
```

### 🔑 **Configurazione Microsoft Foundry**

Accedi con Azure CLI (`az login`) affinché `AzureCliCredential` possa autenticarti, quindi imposta i dettagli del tuo progetto Microsoft Foundry.

**Configurazione dell’Ambiente (file .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

### 🏢 **Casi d’Uso Aziendali**

**Esempi di Processi Aziendali:**
- **Onboarding Cliente**: Workflow multi-step per verifica e configurazione
- **Pipeline di Contenuti**: Creazione, revisione e pubblicazione automatizzata dei contenuti
- **Elaborazione Dati**: Workflow ETL con trasformazioni abilitate AI
- **Controllo Qualità**: Processi di testing e validazione automatizzati

**Vantaggi del Workflow:**
- 🎯 **Affidabilità**: Esecuzione deterministica con recupero errori
- 📈 **Scalabilità**: Gestisci l’automazione di processi ad alto volume
- 🔍 **Osservabilità**: Tracciamenti completi di audit e monitoraggio
- 🔧 **Manutenibilità**: Design visuale e componenti modulari

## 🎨 Modelli di Design per Workflow

### Struttura Base del Workflow
```mermaid
graph TD
    A[Inizio] --> B[Compito Agente 1]
    B --> C{Punto di Decisione}
    C -->|Successo| D[Compito Agente 2]
    C -->|Fallimento| E[Gestore Errori]
    D --> F[Fine]
    E --> F
```

**Componenti Chiave:**
- **WorkflowBuilder**: Motore principale di orchestrazione
- **WorkflowEvent**: Gestione eventi e comunicazione
- **WorkflowViz**: Rappresentazione visuale del workflow e debug

Costruiamo insieme il tuo primo workflow intelligente! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
